# torch.compile: Step-only vs Full Loop

Tests whether compiling the **entire solver loop** (vs just the step function) changes the story.

**Hypothesis**: compiling only `D @ u` gives torch.compile nothing to fuse across iterations.
Compiling the whole loop lets the compiler see all 2773 iterations as one graph,
eliminating Python overhead and potentially pipelining kernel launches.

In [ ]:
import time
import torch

torch.set_default_dtype(torch.float64)
print(f'PyTorch: {torch.__version__}')
print(f'CUDA:    {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:     {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Same parameters as the comparison notebook ────────────────────────────────
L, alpha, T_total = 1.0, 0.00025, 5.0
nx    = 1000
dx    = L / (nx - 1)
r_exp = 0.45                          # CFL safety factor
dt    = r_exp * dx**2 / alpha
nt    = int(T_total / dt) + 1
r     = alpha * dt / dx**2

print(f'Grid: {nx} pts, {nt} steps, r={r:.4f}')

In [ ]:
def build_D(nx, r, device):
    D = torch.zeros(nx, nx, device=device)
    for i in range(1, nx - 1):
        D[i, i-1], D[i, i], D[i, i+1] = r, 1 - 2*r, r
    D[0,  0],  D[0,  1]  = 1 - 2*r, 2*r
    D[-1, -2], D[-1, -1] = 2*r,     1 - 2*r
    return D

# ── Solver variants ───────────────────────────────────────────────────────────
def solve_uncompiled(D, u0, nt):
    u = u0.clone()
    for _ in range(nt):
        u = D @ u
    return u

def _step(D, u):
    return D @ u

def solve_step_compiled(D, u0, nt, step_fn):
    u = u0.clone()
    for _ in range(nt):
        u = step_fn(D, u)
    return u

def solve_full_loop(D, u0, nt):
    """Entire loop in one compiled graph."""
    u = u0.clone()
    for _ in range(nt):
        u = D @ u
    return u

compiled_step       = torch.compile(_step)
compiled_full       = torch.compile(solve_full_loop)
compiled_full_fast  = torch.compile(solve_full_loop, mode='max-autotune')

print('Variants defined.')

In [ ]:
def bench(fn, *args, n_runs=3, warmup=2, cuda=False):
    for _ in range(warmup):
        fn(*args)
        if cuda: torch.cuda.synchronize()
    times = []
    for _ in range(n_runs):
        if cuda: torch.cuda.synchronize()
        t0 = time.perf_counter()
        fn(*args)
        if cuda: torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    return min(times) * 1000   # best-of-n, ms

print('Benchmark helper defined.')

In [ ]:
header = f"{'Method':<38} {'Device':<6} {'ms':>10} {'vs uncompiled':>15}"
sep    = '-' * len(header)

print(header)
print(sep)

device_list = ['cpu'] + (['cuda'] if torch.cuda.is_available() else [])

for device_str in device_list:
    device = torch.device(device_str)
    cuda   = device_str == 'cuda'
    label  = 'GPU' if cuda else 'CPU'

    D  = build_D(nx, r, device)
    x  = torch.linspace(0, L, nx, device=device)
    u0 = torch.exp(-((x - 0.5)**2) / (2 * 0.1**2))

    t_base = bench(solve_uncompiled, D, u0, nt, cuda=cuda)
    print(f"{'Uncompiled':<38} {label:<6} {t_base:>10.1f} {'(baseline)':>15}")

    t = bench(solve_step_compiled, D, u0, nt, compiled_step, cuda=cuda)
    print(f"{'compile(step only)':<38} {label:<6} {t:>10.1f} {t_base/t:>14.2f}x")

    t = bench(compiled_full, D, u0, nt, cuda=cuda)
    print(f"{'compile(full loop)':<38} {label:<6} {t:>10.1f} {t_base/t:>14.2f}x")

    t = bench(compiled_full_fast, D, u0, nt, cuda=cuda)
    print(f"{'compile(full loop, max-autotune)':<38} {label:<6} {t:>10.1f} {t_base/t:>14.2f}x")

    print(sep)